# Physics-Informed Transformer — NFL Big Data Bowl 2026

This notebook is a reviewer-friendly walk-through of the winning HODL submission.
All heavy lifting lives in the `bdb2026` package; this notebook simply imports,
configures, and runs.

**Headline result.** Test RMSE = **1.172 yards** on the held-out Kaggle test set.

**Author.** Judy Zhu (model architecture, training pipeline, results).

## 1. Setup

The package is installable from the repo root with `pip install -e .` (see
`pyproject.toml` / `requirements.txt`). On Colab, you can also run the cell
below first to install dependencies:

In [ ]:
# Colab setup — uncomment if needed
# !pip install -q torch numpy pandas
# !pip install -e ..  # if running from notebooks/ inside the repo

In [ ]:
import torch
import numpy as np

from bdb2026.config import TrainConfig
from bdb2026.train import train

torch.manual_seed(42)
np.random.seed(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. Data

The Big Data Bowl 2026 dataset is provided as 18 weekly CSV pairs:

* `input_2023_w{01..18}.csv`  — pre-throw frames (the model's input window)
* `output_2023_w{01..18}.csv` — post-throw frames (the supervision target)

Place them in `data/raw/` (or pass a different `--data-dir`). We use
**weeks 1-13 for training** and **weeks 14-18 for held-out validation**, mirroring
the Kaggle scoring protocol.

## 3. Configuration

All hyperparameters live in a single `TrainConfig` dataclass. The defaults
below reproduce the submitted model.

In [ ]:
cfg = TrainConfig(
    data_dir='data/raw',
    checkpoint_path='checkpoints/bdb2026_phys_transformer.pt',
    batch_size=128,
    learning_rate=1e-3,
    epochs=30,
    hidden_dim=96,
    num_layers=1,
    num_heads=2,
    lambda_phys=0.003,
)
cfg

## 4. Architecture summary

**Encoder.** Standard transformer encoder over per-frame inputs formed from
11 standardized continuous features + 4 learned categorical embeddings + a
learned time embedding.

**Decoder.** A `GRUCell` consumes `[xy_t, v_t, cat_embed, enc_summary]` at each
future timestep and predicts an *acceleration* `a_t`. Velocity and position
are then integrated explicitly via Euler steps:

$$
v_{t+1} = v_t + a_t \, dt, \qquad xy_{t+1} = xy_t + v_{t+1} \, dt
$$

with a **learnable timestep `dt`** parameterized in log-space. Because the
network outputs *accelerations* rather than raw coordinates, the kinematic
structure is baked into the architecture itself.

**Loss.** Masked RMSE (matching the Kaggle metric) plus a soft physics
regularizer that penalizes predicted speeds above 12 yd/s and accelerations
above 8 yd/s².

## 5. Train

The full training loop (data loading, vocab building, training, validation,
best-checkpoint saving) is wrapped in `bdb2026.train.train`.

In [ ]:
# train(cfg)  # uncomment to actually run; ~30 min on a single Colab GPU

## 6. Results

On a single T4 GPU, training for 30 epochs achieved:

| Split | RMSE (yards) |
| --- | --- |
| Validation (weeks 14-18) | ~1.13 |
| Test (Kaggle held-out)   | **1.172** |

Compared with the other architectures we tried on the HODL team:

| Model | Test RMSE (yards) |
| --- | --- |
| XGBoost | 2.406 |
| Physics-Informed ResNet | 1.290 |
| LSTM | 1.456 |
| Transformer | 1.249 |
| Physics-Informed LSTM | 1.535 |
| **Physics-Informed Transformer (this model)** | **1.172** |

## 7. Inference

To run inference inside the Kaggle environment, load the checkpoint and call
`model(cont, tin_mask, cat, max_t_out)` per play. The Kaggle submission cell is
kept separate because it relies on the Kaggle API and won't execute outside
that environment.